# MD-4 · Similitud textual

TF-IDF, LSA y MiniLM; secciones 2.4 y 3.5.

**Proyecto:** H&M · UNMSM/FISI · Grupo 2 · Modelado + evaluación automatizada en el mismo notebook · Versión de protocolo: 10-11-2026.

**Importante:** las salidas y métricas deben provenir de la ejecución real en Kaggle. No se incluyen cifras simuladas como resultados del proyecto.

**Implementador y auditor:** coordinación del grupo. Código preparado para Kaggle; evaluación de datos reales pendiente hasta ejecutar. Las pruebas humanas de MD-4 no se inventan.


**Instalación condicional.** Ejecutar en Kaggle con Internet habilitado únicamente si faltan paquetes; para Sentence-BERT también se requiere descarga del modelo o un artefacto predescargado.

In [7]:
import importlib.util,subprocess,sys
DEPENDENCIAS = [('duckdb', 'duckdb==1.3.2'), ('sentence_transformers', 'sentence-transformers>=3.0.0'), ('transformers', 'transformers>=4.40.0')]
missing=[pkg for module,pkg in DEPENDENCIAS if importlib.util.find_spec(module) is None]
if missing:
    print('Dependencias faltantes:', missing)
    try: subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
    except Exception as exc:
        raise RuntimeError('No se pudieron instalar dependencias. En Kaggle habilita Internet en Notebook Options, o adjunta ruedas compatibles.') from exc
else:print('Dependencias específicas ya instaladas.')


Dependencias específicas ya instaladas.


## 1. Objetivo y alcance
Evaluar afinidad textual complementaria a taxonomía, con TF-IDF+coseno, LSA+coseno y Sentence-BERT all-MiniLM-L6-v2. MD4-P01…P04; ninguna evaluación PPS puede inventarse. Los textos de Q=30 participan en ajuste global no supervisado, pero no en selección de hiperparámetros.

## 2. Configuración y entorno
Se aprovecha GPU **solo** para embeddings Sentence-BERT. TF-IDF/LSA sobre CSR completa. No se construye similitud N×N.

In [8]:
# Configuración portable: Kaggle primero, pero sin asumir rutas fijas.
from pathlib import Path
import os, json, time, platform, hashlib, warnings, sys
import numpy as np, pandas as pd
from packaging.version import Version
from importlib.metadata import version, PackageNotFoundError
import psutil
DATA_FILENAMES = {'transactions':'transacciones_preparadas.parquet',
                  'customers':'clientes_preparados.parquet',
                  'segmentation':'clientes_vista_segmentacion.parquet',
                  'articles':'articles_preparado.parquet'}
KAGGLE_CANDIDATE = Path('/kaggle/input/datasets/marcocastilla/hm-grupo2-datos-preparados')
INPUT_DIR = Path(os.environ.get('HM_INPUT_DIR', str(KAGGLE_CANDIDATE)))
if not all((INPUT_DIR / x).is_file() for x in DATA_FILENAMES.values()):
    roots=[Path('/kaggle/input'), Path.cwd(), Path('/mnt/data')]
    found=[]
    for root in roots:
        if root.is_dir():
            for path in root.rglob('transacciones_preparadas.parquet'):
                if all((path.parent / fn).is_file() for fn in DATA_FILENAMES.values()): found.append(path.parent.resolve())
    found=list(dict.fromkeys(found))
    if len(found)!=1: raise FileNotFoundError(f'Esperado un único directorio con los cuatro Parquet. Encontrados: {found}. Defina HM_INPUT_DIR.')
    INPUT_DIR=found[0]
FILES={k:INPUT_DIR / v for k,v in DATA_FILENAMES.items()}
OUTPUT_ROOT=Path(os.environ.get('HM_OUTPUT_DIR','/kaggle/working/hm_modelamiento' if Path('/kaggle/working').is_dir() else str(Path.cwd()/'hm_modelamiento')))
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print('Datos:',INPUT_DIR,'RAM disponible GiB:',round(psutil.virtual_memory().available/2**30,2))
print('CPU lógicas:',os.cpu_count(),'Python:',sys.version.split()[0])
print({k:(str(v), round(v.stat().st_size/2**20,1)) for k,v in FILES.items()})
def _has_pkg(n):
    try:version(n);return True
    except PackageNotFoundError:return False
try:
    import duckdb
except ImportError as exc:
    raise RuntimeError('Falta duckdb. En Kaggle: habilitar instalación si es necesaria y ejecutar %pip install duckdb==1.3.2') from exc
con=duckdb.connect(database=':memory:')
con.execute('PRAGMA threads=4')
con.execute("SET memory_limit='8GB'")
PTRANS=str(FILES['transactions']).replace("'", "''")
PART=str(FILES['articles']).replace("'", "''")
PCUST=str(FILES['customers']).replace("'", "''")
PSEG=str(FILES['segmentation']).replace("'", "''")

OUT=OUTPUT_ROOT/'md4';OUT.mkdir(parents=True,exist_ok=True)
def save_csv(df, filename):
    p=OUT/filename; p.parent.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(df).to_csv(p,index=False); print('Guardado:',p); return p

def save_json(obj, filename):
    p=OUT/filename; p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    print('Guardado:',p);return p

def sha256_path(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

SOURCE={'sha256':{k:sha256_path(v) for k,v in FILES.items()},'runtime':platform.platform(),'python':sys.version.split()[0],
        'packages':{n:(version(n) if _has_pkg(n) else None) for n in ['numpy','pandas','scikit-learn','statsmodels','duckdb']}}
save_json(SOURCE,'entrada_y_entorno.json')
STATUS=[]
def registro(prueba,etapa,estado,detalle='',duracion_s=None):
    item={'prueba':prueba,'etapa':etapa,'estado':estado,'detalle':str(detalle),'duracion_s':duracion_s}
    STATUS.append(item)
    pd.DataFrame(STATUS).to_csv(OUT/'estado_experimentos.csv',index=False)
    return item


Datos: /kaggle/input/datasets/marcocastilla/hm-grupo2-datos-preparados RAM disponible GiB: 29.06
CPU lógicas: 4 Python: 3.13.16
{'transactions': ('/kaggle/input/datasets/marcocastilla/hm-grupo2-datos-preparados/transacciones_preparadas.parquet', 448.3), 'customers': ('/kaggle/input/datasets/marcocastilla/hm-grupo2-datos-preparados/clientes_preparados.parquet', 160.2), 'segmentation': ('/kaggle/input/datasets/marcocastilla/hm-grupo2-datos-preparados/clientes_vista_segmentacion.parquet', 80.4), 'articles': ('/kaggle/input/datasets/marcocastilla/hm-grupo2-datos-preparados/articles_preparado.parquet', 16.9)}
Guardado: /kaggle/working/hm_modelamiento/md4/entrada_y_entorno.json


In [9]:

# Cache cannot be silently reused with a different set of input files.
INPUT_SIGNATURE = SOURCE['sha256']
p_input = OUT / 'origen_cache.json'
if p_input.exists():
    old = json.loads(p_input.read_text(encoding='utf-8'))
    if old.get('sha256') != INPUT_SIGNATURE:
        raise RuntimeError('El origen cambió respecto a los cachés previos. Usar un nuevo HM_OUTPUT_DIR o depurar cachés de forma controlada.')
else:
    p_input.write_text(json.dumps({'sha256':INPUT_SIGNATURE},indent=2),encoding='utf-8')


## 3. Entrada, integridad y estratos (MD4-P01)
105.542 artículos, 416 imputados, 43.404 descripciones originales distintas. Auditar longitud real en WordPieces. No deduplicar TF-IDF/LSA antes del ajuste.

In [10]:
from scipy.sparse import save_npz,load_npz,csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from joblib import dump,load
from scipy.stats import entropy
SRC=con.execute(f"""SELECT article_id,product_code,prod_name,detail_desc_raw,
 detail_desc_clean,detail_desc_imputada,corpus_texto_nlp,product_group_name,
 product_type_no,product_type_name,department_no,department_name,
 colour_group_name,graphical_appearance_name,has_sales
 FROM read_parquet('{PART}') ORDER BY article_id""").df()
assert len(SRC)==SRC.article_id.nunique()==105542
SRC['text']=SRC.detail_desc_clean.fillna('').astype(str)
SRC['imputado']=SRC.detail_desc_imputada.astype(bool)
SRC['tipo']=SRC.product_type_name.fillna('Unknown').astype(str)
assert int(SRC.imputado.sum())==416
assert SRC.text.str.len().gt(0).all()
assert SRC.loc[~SRC.imputado,'detail_desc_raw'].nunique()==43404
SRC['n_palabras']=SRC.text.str.split().str.len()
counts=SRC.tipo.value_counts();top10=set(counts.nlargest(10).index)
def estrato(tipo):
    if tipo in top10:return 'A'
    n=counts[tipo]
    if n>=100:return 'B'
    return 'C'
SRC['estrato']=SRC.tipo.map(estrato)
ELIG=SRC[(~SRC.imputado)&(SRC.n_palabras>=3)&(SRC.tipo!='Unknown')].copy()
print('Estratos artículos:',SRC.estrato.value_counts().to_dict(),
     'tipos elegibles:',ELIG.groupby('estrato').tipo.nunique().to_dict())
assert ELIG.groupby('estrato').tipo.nunique().min()>=10, 'No se pueden elegir 10 tipos distintos por estrato'
registro('MD4-P01','integridad_corpus','PASS',{'n':len(SRC),'imputados':SRC.imputado.sum()})


Estratos artículos: {'A': 61146, 'B': 43151, 'C': 1245} tipos elegibles: {'A': 10, 'B': 55, 'C': 65}


{'prueba': 'MD4-P01',
 'etapa': 'integridad_corpus',
 'estado': 'PASS',
 'detalle': "{'n': 105542, 'imputados': np.int64(416)}",
 'duracion_s': None}

## 4. Muestreo congelado de consultas
170 consultas de selección y 30 retenidas, distribuidas 10 por estrato y como máximo una por tipo dentro de cada estrato. Semilla 20261011; muestreo no debe depender de métricas de test.

In [11]:
rng=np.random.default_rng(20261011)
# Elegir 10 tipos por estrato, 1 artículo de cada uno. Los otros 170 se toman sin intersección.
held=[]
for group in ('A','B','C'):
    types=np.array(sorted(ELIG.loc[ELIG.estrato==group,'tipo'].unique()))
    chosen_types=rng.choice(types,size=10,replace=False)
    for typ in chosen_types:
        candidates=ELIG.index[ELIG.tipo==typ].to_numpy()
        held.append(int(rng.choice(candidates)))
held=np.asarray(held,dtype=int)
remaining=ELIG.index.difference(held).to_numpy(dtype=int)
calibration=rng.choice(remaining,size=170,replace=False)
assert len(held)==30 and len(calibration)==170 and not set(held)&set(calibration)
save_csv(SRC.loc[held,['article_id','tipo','estrato','imputado']],'md4_consultas_holdout.csv')
save_csv(SRC.loc[calibration,['article_id','tipo','estrato','imputado']],'md4_consultas_ajuste.csv')
# Los 30 article_id de Q se mantienen fuera de TODA selección de hiperparámetros.


Guardado: /kaggle/working/hm_modelamiento/md4/md4_consultas_holdout.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_consultas_ajuste.csv


PosixPath('/kaggle/working/hm_modelamiento/md4/md4_consultas_ajuste.csv')

## 5. Tokenización real y truncamiento MiniLM
El contador de WordPieces debe ejecutarse con el tokenizer exacto y sin truncamiento para obtener longitudes completas, incluyendo tokens especiales.

In [12]:
# MD-4 opcional — REEMPLAZAR la celda de tokenizer antes de futuras ejecuciones desde cero.
# No es necesario ejecutar esta celda en la sesión donde MD-4 ya terminó.
from pathlib import Path
import os, json
import numpy as np
import torch
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

MODEL_ID='sentence-transformers/all-MiniLM-L6-v2'
EMBED_FILE=Path(OUT)/'sbert_embeddings.npy'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('CUDA disponible:',torch.cuda.is_available(),'Dispositivo:',DEVICE)
if DEVICE=='cuda':print('GPU:',torch.cuda.get_device_name(0))
if DEVICE!='cuda' and not EMBED_FILE.exists() and os.environ.get('MD4_PERMITIR_CPU')!='1':
    raise RuntimeError('SBERT sin GPU: configura Settings > Accelerator > GPU T4x2 en Kaggle ANTES de Run All. '
                       'La inferencia CPU se permite solo con MD4_PERMITIR_CPU=1 explícito.')

# Token HF opcional. Guardarlo en Kaggle Add-ons/Secrets como HF_TOKEN, nunca escribirlo en outputs.
HF_TOKEN=os.getenv('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN=None
print('Autenticación Hugging Face:', 'configurada' if HF_TOKEN else 'anónima (advertencia no fatal)')

texts=SRC.text.to_numpy()
unique_texts,inverse=np.unique(texts,return_inverse=True)
path_tokens=Path(OUT)/'md4_tokenizador.json'
if path_tokens.is_file():
    cached=json.loads(path_tokens.read_text(encoding='utf-8'))
    if cached.get('unique_texts')==len(unique_texts) and 'max_tokens' in cached:
        TOK_AUDIT=cached
        print('Auditoría de tokens reutilizada:',cached)
    else:
        TOK_AUDIT=None
else:
    TOK_AUDIT=None
if TOK_AUDIT is None:
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True,token=HF_TOKEN)
    lengths=np.empty(len(unique_texts),dtype='int32')
    for inicio in range(0,len(unique_texts),256):
        b=unique_texts[inicio:inicio+256].tolist()
        encoded=tokenizer(b,add_special_tokens=True,truncation=False,padding=False)
        lengths[inicio:inicio+len(b)]=[len(v) for v in encoded['input_ids']]
    lens_full=lengths[inverse]
    TOK_AUDIT={'max_tokens':int(lengths.max()),'unique_texts':len(unique_texts),
               'articulos_mas_256':int((lens_full>256).sum()),
               'textos_unicos_mas_256':int((lengths>256).sum())}
    save_json(TOK_AUDIT,'md4_tokenizador.json')
registro('MD4-P01','wordpieces','PASS',TOK_AUDIT)


CUDA disponible: True Dispositivo: cuda
GPU: Tesla T4
Autenticación Hugging Face: anónima (advertencia no fatal)
Auditoría de tokens reutilizada: {'max_tokens': 168, 'unique_texts': 43779, 'articulos_mas_256': 0, 'textos_unicos_mas_256': 0}


{'prueba': 'MD4-P01',
 'etapa': 'wordpieces',
 'estado': 'PASS',
 'detalle': "{'max_tokens': 168, 'unique_texts': 43779, 'articulos_mas_256': 0, 'textos_unicos_mas_256': 0}",
 'duracion_s': None}

In [13]:

AUDIT=[{'campo':'N_articulos','valor':len(SRC)},
       {'campo':'N_descripciones_imputadas','valor':int(SRC.imputado.sum())},
       {'campo':'N_desc_raw_unicas_originales','valor':SRC.loc[~SRC.imputado,'detail_desc_raw'].nunique()},
       {'campo':'N_desc_clean_unicas','valor':SRC.text.nunique()},
       {'campo':'N_corpus_texto_nlp_unicos','valor':SRC.corpus_texto_nlp.nunique()},
       {'campo':'max_wordpieces_unicos','valor':TOK_AUDIT['max_tokens']},
       {'campo':'n_articulos_mayor_256_wordpieces','valor':TOK_AUDIT['articulos_mas_256']}]
for label,g in SRC.groupby('estrato'):
    AUDIT.append({'campo':'articulos_estrato_'+label,'valor':len(g)})
    AUDIT.append({'campo':'tipos_estrato_'+label,'valor':g.tipo.nunique()})
for label,g in ELIG.groupby('estrato'):
    AUDIT.append({'campo':'articulos_elegibles_'+label,'valor':len(g)})
    AUDIT.append({'campo':'tipos_elegibles_'+label,'valor':g.tipo.nunique()})
for label,flag in [('E1_imputados',SRC.imputado),('E2_menos3',SRC.n_palabras<3),('E3_unknown',SRC.tipo=='Unknown')]:
    AUDIT.append({'campo':label,'valor':int(flag.sum())})
save_csv(AUDIT,'md4_auditoria_corpus.csv')
# Text-index mapping avoids including the raw description multiple times.
pd.DataFrame({'article_id':SRC.article_id,'id_texto_unico':inverse}).to_csv(
   OUT/'md4_mapa_articulos_textos.csv',index=False)
registro('MD4-P01','auditoria_completa','PASS',len(AUDIT))


Guardado: /kaggle/working/hm_modelamiento/md4/md4_auditoria_corpus.csv


{'prueba': 'MD4-P01',
 'etapa': 'auditoria_completa',
 'estado': 'PASS',
 'detalle': '22',
 'duracion_s': None}

## 6. Representaciones principales y búsqueda top-k
Ajuste de cuatro TF-IDF **sobre los 105.542 artículos**, dos LSA sobre el TF-IDF elegido y Sentence-BERT sobre cadenas únicas. Recuperación adaptativa: 250 → 1.000 → 5.000 → todos los vecinos con coseno positivo, excluyendo ID, product_code y texto idéntico.

In [14]:
def tfidf_fit(strings,stopwords,ngrams):
    vec=TfidfVectorizer(ngram_range=ngrams,stop_words=stopwords,min_df=2,max_df=.95,
                        sublinear_tf=True,norm='l2',dtype=np.float32)
    return vec,vec.fit_transform(strings).tocsr()

def rank_one(query_i,matrix,allowed=None,top=10,exclude_identical=True):
    N=matrix.shape[0]
    if hasattr(matrix,'tocsr'):
        sim=(matrix[query_i]@matrix.T).toarray().ravel()
    else:
        sim=np.asarray(matrix@matrix[query_i],dtype='float32').ravel()
    sim=np.asarray(sim,dtype='float32');sim[query_i]=-np.inf
    candidate_mask=(SRC.product_code.to_numpy()!=SRC.product_code.iloc[query_i]) & (sim>0) & np.isfinite(sim)
    if exclude_identical:candidate_mask &= (texts!=texts[query_i])
    if allowed is not None:candidate_mask &= np.asarray(allowed,dtype=bool)
    # Total deterministic ranking; the adaptive 250/1000/5000/N windows are
    # inspected in order WITHOUT leaking exclusions into candidate ordering.
    order=np.lexsort((np.arange(N),-sim))
    for window in (250,1000,5000,N):
        cand=order[:min(window,N)]
        good=cand[candidate_mask[cand]]
        if len(good)>=top or window==N:
            return [(int(i),float(sim[i])) for i in good[:top]]
    return []

def batch_ranking(qindices,matrix,allowed=None,top=10):
    # Cada consulta crea únicamente un vector de N similitudes, nunca N*N.
    return {int(q):rank_one(int(q),matrix,allowed,top) for q in qindices}

def rank_stats(ranks,ks=(5,10)):
    out=[]
    for k in ks:
        total=k*len(ranks)
        filled=sum(min(k,len(v)) for v in ranks.values())
        out.append({'k':k,'TCC_at_k':filled/total,'vacantes':total-filled,'denominador':total})
    return out

TF_RESULTS={}
for sw,ng in [(None,(1,1)),(None,(1,2)),('english',(1,1)),('english',(1,2))]:
    label=f"TF_{'sinSW' if sw is None else 'SW'}_{ng[1]}"
    vec,mat=tfidf_fit(texts,sw,ng)
    rr=batch_ranking(calibration,mat)
    # Entropía normalizada de contribuciones léxicas en los top-5, mayor = menor concentración.
    ents=[]
    for qi,v in rr.items():
        for ni,_ in v[:5]:
            active=mat[qi].multiply(mat[ni]).data
            if active.size>1 and active.sum()>0:
                p=active/active.sum();ents.append(float(entropy(p)/np.log(len(active))))
            elif active.size==1:ents.append(0.)
    cover=rank_stats(rr)[1]['TCC_at_k']
    TF_RESULTS[label]={'vec':vec,'mat':mat,'coverage':cover,'entropy':float(np.mean(ents)) if ents else np.nan,
                       'ranks':rr,'sw':sw,'ng':ng}
    print(label,'vocab',mat.shape[1],'TCC@10',cover,'entropy',TF_RESULTS[label]['entropy'])
# Si ≥2 están empatados hasta un punto porcentual, preferir la canónica aprobada.
score=max(x['coverage'] for x in TF_RESULTS.values())
near=[n for n,x in TF_RESULTS.items() if score-x['coverage']<.01]
canonical='TF_SW_2'
if canonical in near:CHOSEN_TF=canonical
else:CHOSEN_TF=sorted(near,key=lambda n:(-TF_RESULTS[n]['entropy'],n))[0]
base=TF_RESULTS[CHOSEN_TF]
save_csv([{'modelo':n,'tcc10':v['coverage'],'entropia':v['entropy']} for n,v in TF_RESULTS.items()],'md4_seleccion_tfidf.csv')
print('TF-IDF congelado:',CHOSEN_TF)
registro('MD4-P02','seleccion_tfidf','PASS',CHOSEN_TF)

TF_sinSW_1 vocab 3063 TCC@10 1.0 entropy 0.8769889663666058
TF_sinSW_2 vocab 30755 TCC@10 1.0 entropy 0.9105373578956614
TF_SW_1 vocab 2921 TCC@10 1.0 entropy 0.9150283605703226
TF_SW_2 vocab 36949 TCC@10 1.0 entropy 0.9179560906078448
Guardado: /kaggle/working/hm_modelamiento/md4/md4_seleccion_tfidf.csv
TF-IDF congelado: TF_SW_2


{'prueba': 'MD4-P02',
 'etapa': 'seleccion_tfidf',
 'estado': 'PASS',
 'detalle': 'TF_SW_2',
 'duracion_s': None}

In [15]:
import threading

def svd_with_rss_monitor(fit_fn):
    process=psutil.Process(os.getpid())
    rss_start=process.memory_info().rss
    peak=[rss_start]
    stop=threading.Event()
    def watch():
        while not stop.is_set():
            try:peak[0]=max(peak[0],process.memory_info().rss)
            except psutil.NoSuchProcess:break
            stop.wait(0.05)
    watcher=threading.Thread(target=watch,daemon=True)
    watcher.start()
    try:result=fit_fn()
    finally:
        stop.set();watcher.join(timeout=1)
        peak[0]=max(peak[0],process.memory_info().rss)
    return result, (peak[0]-rss_start)/2**30
LSA_RESULTS={}

for d in (100,200):
    tt=time.monotonic();svd=TruncatedSVD(n_components=d,random_state=20261011)
    raw_svd,ram_peak_delta_gib=svd_with_rss_monitor(lambda:svd.fit_transform(base['mat']))
    dense=normalize(raw_svd.astype('float32'))
    rr=batch_ranking(calibration,dense)
    jac=[]
    for q in calibration:
        aa={i for i,_ in base['ranks'][int(q)]};bb={i for i,_ in rr[int(q)]}
        jac.append(len(aa&bb)/len(aa|bb) if (aa or bb) else 0)
    LSA_RESULTS[d]={'model':svd,'matrix':dense,'ranks':rr,
          'varianza':float(svd.explained_variance_ratio_.sum()),'jaccard':float(np.mean(jac)),
          'seconds':time.monotonic()-tt,'rss_peak_delta_gib':ram_peak_delta_gib}
    print('LSA',d,'varianza=',LSA_RESULTS[d]['varianza'],'Jaccard=',LSA_RESULTS[d]['jaccard'])
possible=[d for d in (100,200) if LSA_RESULTS[d]['varianza']>=.25 and .30<=LSA_RESULTS[d]['jaccard']<=.85]
CHOSEN_LSA=min(possible) if possible else 200
save_csv([{'dimensiones':d,**{k:v for k,v in x.items() if k in ('varianza','jaccard','seconds','rss_peak_delta_gib')}} for d,x in LSA_RESULTS.items()], 'md4_seleccion_lsa.csv')
registro('MD4-P02','seleccion_lsa','PASS',CHOSEN_LSA)
# Consolidado exigido para modelamiento y evaluación Semana 12.
TF_HYPERS=[]
for name,x in TF_RESULTS.items():
    TF_HYPERS.append({'familia':'TFIDF','modelo':name,'stop_words':str(x['sw']),
      'ngram_range':str(x['ng']),'min_df':2,'max_df':.95,'sublinear_tf':True,
      'n_componentes':np.nan,'TCC_at_10':x['coverage'],'entropia':x['entropy'],
      'varianza':np.nan,'jaccard_TFIDF':np.nan,'seleccionado':name==CHOSEN_TF})
for dim,x in LSA_RESULTS.items():
    TF_HYPERS.append({'familia':'LSA','modelo':f'LSA_{dim}',
      'stop_words':str(base['sw']),'ngram_range':str(base['ng']),
      'min_df':2,'max_df':.95,'sublinear_tf':True,'n_componentes':dim,
      'TCC_at_10':np.nan,'entropia':np.nan,'varianza':x['varianza'],
      'jaccard_TFIDF':x['jaccard'],'seleccionado':dim==CHOSEN_LSA})
save_csv(TF_HYPERS,'md4_seleccion_hiperparametros.csv')



LSA 100 varianza= 0.30418357253074646 Jaccard= 0.25511106843742065
LSA 200 varianza= 0.3988111913204193 Jaccard= 0.3090437441511927
Guardado: /kaggle/working/hm_modelamiento/md4/md4_seleccion_lsa.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_seleccion_hiperparametros.csv


PosixPath('/kaggle/working/hm_modelamiento/md4/md4_seleccion_hiperparametros.csv')

In [16]:
# MD-4 opcional — REEMPLAZAR la celda SBERT antes de futuras ejecuciones desde cero.
# Reutiliza embeddings ya calculados sin descargar pesos otra vez.
from pathlib import Path
import numpy as np, os, torch
EMBED_FILE=Path(OUT)/'sbert_embeddings.npy'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
REUTILIZADO=EMBED_FILE.is_file()
if REUTILIZADO:
    SBERT=np.load(EMBED_FILE,mmap_mode='r')
    if SBERT.shape!=(len(SRC),384) or SBERT.dtype!=np.dtype('float32'):
        raise ValueError('El cache SBERT no coincide con corpus y dimensión 384; revisar antes de reutilizar.')
    print('SBERT: embeddings reutilizados desde cache, sin descargar ni inferir.')
else:
    if DEVICE!='cuda' and os.environ.get('MD4_PERMITIR_CPU')!='1':
        raise RuntimeError('GPU no disponible. Evita el cálculo accidental en CPU: activa T4x2 y reinicia.')
    encoder=SentenceTransformer(MODEL_ID,device=DEVICE,token=HF_TOKEN if HF_TOKEN else None)
    print('SBERT inferencia en',DEVICE, 'con',len(unique_texts),'textos únicos',flush=True)
    # Batch moderado para evitar OOM; 64 es suficiente en T4 y reproducible.
    emb_u=encoder.encode(unique_texts.tolist(),batch_size=64,
        show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True).astype('float32')
    SBERT=emb_u[inverse]
    np.save(EMBED_FILE,SBERT)
assert SBERT.shape==(len(SRC),384)
registro('MD4-P02','embeddings_sbert','PASS',{'shape':SBERT.shape,'device':DEVICE,
    'origen':'cache' if REUTILIZADO else 'inferido'})


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT inferencia en cuda con 43779 textos únicos


Batches:   0%|          | 0/685 [00:00<?, ?it/s]

{'prueba': 'MD4-P02',
 'etapa': 'embeddings_sbert',
 'estado': 'PASS',
 'detalle': "{'shape': (105542, 384), 'device': 'cuda', 'origen': 'inferido'}",
 'duracion_s': None}

## 7. Evaluación estructural retenida y controles de imputación MD4-P02/P03
Consulta hold-out solo después de congelar hiperparámetros. Regímenes: catálogo completo, retirar imputados con pesos fijos y refit en originales. Los indicadores TCT, TCMG, TDIG, TCTipo y TDIT requieren una definición operacional formal; aquí se exportan conteos no ambiguos y sus denominadores, sin inventar fórmulas.

In [17]:
orig_allowed=(~SRC.imputado).to_numpy()
MODEL_REPS={'TFIDF':base['mat'],'LSA':LSA_RESULTS[CHOSEN_LSA]['matrix'],'SBERT':SBERT}
SEARCH_RESULTS={};ROW_RESULTS=[]
for name,rep in MODEL_REPS.items():
    for scenario,mask in [('completo',None),('sin_imputados_modelo_fijo',orig_allowed)]:
        ranks=batch_ranking(held,rep,mask)
        SEARCH_RESULTS[(name,scenario)]=ranks
        for k in (5,10):
            for q,v in ranks.items():
                kept=v[:k]
                for rank,(j,score) in enumerate(kept,1):
                    ROW_RESULTS.append({'modelo':name,'regimen':scenario,'k':k,'consulta':SRC.article_id.iloc[q],
                         'vecino':SRC.article_id.iloc[j], 'rango':rank,'coseno':score,
                         'mismo_macrogrupo':bool(SRC.product_group_name.iloc[q]==SRC.product_group_name.iloc[j]),
                         'mismo_departamento':bool(SRC.department_no.iloc[q]==SRC.department_no.iloc[j]),
                         'mismo_tipo':bool(SRC.tipo.iloc[q]==SRC.tipo.iloc[j]),
                         'vecino_imputado':bool(SRC.imputado.iloc[j])})
# Refit TF-IDF y LSA sobre N_orig: preservar el universo ORIGINAL de 105126 para contar df y SVD.
orig_strings=texts[orig_allowed]
orig_idx=np.flatnonzero(orig_allowed)
vec_o,mat_o=tfidf_fit(orig_strings,base['sw'],base['ng'])
lsa_o=TruncatedSVD(n_components=CHOSEN_LSA,random_state=20261011)
lsa_o_mat=normalize(lsa_o.fit_transform(mat_o).astype('float32'))
# Para conservar el mapeo global por article_id, ubicar resultados en las filas originales.
mat_full=vec_o.transform(texts).tocsr()
lsa_full=normalize(lsa_o.transform(mat_full).astype('float32'))
for name,rep in [('TFIDF',mat_full),('LSA',lsa_full),('SBERT',SBERT)]:
    ranks=batch_ranking(held,rep,orig_allowed)
    SEARCH_RESULTS[(name,'refit_orig' if name!='SBERT' else 'sin_imputados_modelo_fijo')]=ranks
    if name=='SBERT':continue  # preentrenado sin ajuste corpus, su régimen ya existe
    for k in (5,10):
        for q,v in ranks.items():
            for rank,(j,score) in enumerate(v[:k],1):
                ROW_RESULTS.append({'modelo':name,'regimen':'refit_orig','k':k,
                    'consulta':SRC.article_id.iloc[q],'vecino':SRC.article_id.iloc[j],
                    'rango':rank,'coseno':score,
                    'mismo_macrogrupo':bool(SRC.product_group_name.iloc[q]==SRC.product_group_name.iloc[j]),
                    'mismo_departamento':bool(SRC.department_no.iloc[q]==SRC.department_no.iloc[j]),
                    'mismo_tipo':bool(SRC.tipo.iloc[q]==SRC.tipo.iloc[j]),
                    'vecino_imputado':bool(SRC.imputado.iloc[j])})
ranking_df=pd.DataFrame(ROW_RESULTS)
save_csv(ranking_df,'md4_comparacion_topk_modelos.csv')
agg=ranking_df.groupby(['modelo','regimen','k']).agg(
    vecinos=('vecino','size'), macrogrupo_igual=('mismo_macrogrupo','sum'),
    departamento_igual=('mismo_departamento','sum'),tipo_igual=('mismo_tipo','sum')).reset_index()
agg['denominador']=agg['k']*30
agg['vacantes']=agg.denominador-agg.vecinos
save_csv(agg,'md4_aporte_vs_jerarquia_conteos.csv')
registro('MD4-P03','estructurales','PARCIAL','Conteos auditables: nombres exactos de las tasas pendientes de definición')


Guardado: /kaggle/working/hm_modelamiento/md4/md4_comparacion_topk_modelos.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_aporte_vs_jerarquia_conteos.csv


{'prueba': 'MD4-P03',
 'etapa': 'estructurales',
 'estado': 'PARCIAL',
 'detalle': 'Conteos auditables: nombres exactos de las tasas pendientes de definición',
 'duracion_s': None}

## 8. Evaluación humana PPS y Kappa
Generar hoja ciega de pares únicos y asignaciones por posición. Nivel 0 = falso positivo/vacante; 1 = pertinente intra-tipo; 2 = pertinente inter-tipo. Se necesita revisión humana genuina; no generar etiquetas con reglas automáticas.

## 8A. Tasas estructurales y diccionario de fórmulas
Los acrónimos no incluyen fórmulas inequívocas en el DOCX: las definiciones operativas aquí se señalan como propuestas, no como hechos aprobados.

In [18]:

# Operational definitions required because the DOCX names TCT/TCMG/TDIG/TCTipo/TDIT
# but does not give their mathematical numerator and denominator. These must be
# ratified before submission; none is presented as an approved definition.
METRIC_DEFINITIONS={
 'TCC@k':'recuperados/(consultas*k); posiciones vacantes cuentan en denominador',
 'TCT@k':'pares con intersección no vacía de palabras normalizadas/(consultas*k); PROPUESTA',
 'TCMG@k':'pares del mismo product_group_name/(consultas*k); PROPUESTA',
 'TDIG@k':'pares de distinto product_group_name/(consultas*k); PROPUESTA',
 'TCTipo@k':'pares del mismo product_type_name/(consultas*k); PROPUESTA',
 'TDIT@k':'pares de distinto product_type_name/(consultas*k); PROPUESTA',
 'Jaccard@k':'interseccion/union de conjuntos top-k article_id para cada consulta; indefinido si ambos vacíos'
}
save_json({'estado':'DEFINICIONES_OPERACIONALES_PROPUESTAS_NO_APROBADAS',
           'formulas':METRIC_DEFINITIONS},'md4_diccionario_metricas.json')
# The comparison table contains a row per real neighbor, no fabricated neighbors.
wordsets={str(row.article_id):set(str(row.text).split()) for row in SRC.itertuples()}
ranking_df['coincidencia_terminos']=ranking_df.apply(lambda r:
    bool(wordsets[str(r['consulta'])] & wordsets[str(r['vecino'])]),axis=1)

def structural_table(rank_frame,qcount=30):
    results=[]
    for (m,reg,k),g in rank_frame.groupby(['modelo','regimen','k']):
        D=int(k*qcount);real=int(len(g))
        results.append({'modelo':m,'regimen':reg,'k':int(k),'D':D,
         'recuperados':real,'vacantes':D-real,'TCC_at_k':real/D,
         'TCT_at_k_PROPUESTA':g.coincidencia_terminos.sum()/D,
         'TCMG_at_k_PROPUESTA':g.mismo_macrogrupo.sum()/D,
         'TDIG_at_k_PROPUESTA':(real-int(g.mismo_macrogrupo.sum()))/D,
         'TCTipo_at_k_PROPUESTA':g.mismo_tipo.sum()/D,
         'TDIT_at_k_PROPUESTA':(real-int(g.mismo_tipo.sum()))/D,
         'tasa_mismo_departamento':g.mismo_departamento.sum()/D,
         'tasa_intertipo':(real-int(g.mismo_tipo.sum()))/D,
         'estado_definiciones':'PROPUESTAS_PENDIENTES_RATIFICACION'})
    return pd.DataFrame(results)

struct=structural_table(ranking_df)
save_csv(struct,'md4_aporte_vs_jerarquia.csv')
# Model-level ranking overlaps: same hold-out queries, model pair and regime only.
OVERLAPS=[]
for regime in ('completo','sin_imputados_modelo_fijo'):
    for a,b in [('TFIDF','LSA'),('TFIDF','SBERT'),('LSA','SBERT')]:
        ra=SEARCH_RESULTS.get((a,regime));rb=SEARCH_RESULTS.get((b,regime))
        if ra is None or rb is None:continue
        for k in (5,10):
            vals=[]
            for qi in held:
                aa={j for j,_ in ra[int(qi)][:k]};bb={j for j,_ in rb[int(qi)][:k]}
                vals.append(len(aa&bb)/len(aa|bb) if (aa or bb) else np.nan)
            OVERLAPS.append({'regimen':regime,'modelo_A':a,'modelo_B':b,'k':k,
                'jaccard_media':float(np.nanmean(vals)) if np.isfinite(vals).any() else np.nan,
                'consultas':len(held)})
save_csv(OVERLAPS,'md4_solapamiento_modelos.csv')
registro('MD4-P03','jerarquia_y_cobertura','PASS_CON_RESERVA',
    'Todas las tablas estructurales calculadas; acrónimos sujetos a ratificación de fórmula')


Guardado: /kaggle/working/hm_modelamiento/md4/md4_diccionario_metricas.json
Guardado: /kaggle/working/hm_modelamiento/md4/md4_aporte_vs_jerarquia.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_solapamiento_modelos.csv


{'prueba': 'MD4-P03',
 'etapa': 'jerarquia_y_cobertura',
 'estado': 'PASS_CON_RESERVA',
 'detalle': 'Todas las tablas estructurales calculadas; acrónimos sujetos a ratificación de fórmula',
 'duracion_s': None}

In [19]:
from sklearn.metrics import cohen_kappa_score
import math
PPS=ranking_df[(ranking_df.k==5)&(ranking_df.regimen=='completo')].copy()
pairs=PPS[['consulta','vecino']].drop_duplicates().sort_values(['consulta','vecino']).reset_index(drop=True)
pairs['id_par']=['Q'+str(i).zfill(4) for i in range(1,len(pairs)+1)]
lookup=SRC.set_index('article_id')
pairs['nombre_consulta']=pairs.consulta.map(lookup['prod_name'])
pairs['descripcion_consulta']=pairs.consulta.map(lookup['detail_desc_clean'])
pairs['tipo_consulta']=pairs.consulta.map(lookup['product_type_name'])
pairs['nombre_vecino']=pairs.vecino.map(lookup['prod_name'])
pairs['descripcion_vecino']=pairs.vecino.map(lookup['detail_desc_clean'])
pairs['tipo_vecino']=pairs.vecino.map(lookup['product_type_name'])
pairs['pps_evaluador_1']='';pairs['pps_evaluador_2']='';pairs['pps_arbitraje']=''
U=len(pairs); n_second=math.ceil(.30*U)
seed_rng=np.random.default_rng(20261011)
first_indices=np.sort(seed_rng.choice(np.arange(U),size=n_second,replace=False))
remaining_indices=np.setdiff1d(np.arange(U),first_indices)
second_indices=np.sort(seed_rng.choice(remaining_indices,size=min(n_second,len(remaining_indices)),replace=False))
pairs['seleccion_primera_ronda_30pct']=False
pairs.loc[first_indices,'seleccion_primera_ronda_30pct']=True
pairs['seleccion_segunda_ronda_si_kappa_menor_060']=False
pairs.loc[second_indices,'seleccion_segunda_ronda_si_kappa_menor_060']=True

# All scenarios require first-review scores for any previously unseen pair.
# Only Q-original real pairs enter Cohen Kappa sampling, per the DOCX.
all_actual=ranking_df[ranking_df.k==5][['consulta','vecino']].drop_duplicates()
extras=all_actual.merge(pairs[['consulta','vecino']],on=['consulta','vecino'],how='left',indicator=True)
extras=extras.loc[extras['_merge']=='left_only',['consulta','vecino']].reset_index(drop=True)
extras['id_par']=['V'+str(i).zfill(4) for i in range(1,len(extras)+1)]
extras['nombre_consulta']=extras.consulta.map(lookup['prod_name'])
extras['descripcion_consulta']=extras.consulta.map(lookup['detail_desc_clean'])
extras['tipo_consulta']=extras.consulta.map(lookup['product_type_name'])
extras['nombre_vecino']=extras.vecino.map(lookup['prod_name'])
extras['descripcion_vecino']=extras.vecino.map(lookup['detail_desc_clean'])
extras['tipo_vecino']=extras.vecino.map(lookup['product_type_name'])
for score_name in ('pps_evaluador_1','pps_evaluador_2','pps_arbitraje'):extras[score_name]=''
extras['seleccion_primera_ronda_30pct']=False
extras['seleccion_segunda_ronda_si_kappa_menor_060']=False
allpairs=pd.concat([pairs,extras],ignore_index=True)
assert allpairs[['consulta','vecino']].duplicated().sum()==0
save_csv(allpairs,'md4_plantilla_pps_todos_escenarios.csv')
save_csv(pairs,'md4_plantilla_pps_ciega.csv')
save_csv(pairs.iloc[first_indices][['id_par','consulta','vecino','nombre_consulta',
    'descripcion_consulta','tipo_consulta','nombre_vecino','descripcion_vecino',
    'tipo_vecino','pps_evaluador_2']],'md4_primera_ronda_segundo_evaluador.csv')
save_csv(pairs.iloc[second_indices][['id_par','consulta','vecino','nombre_consulta',
    'descripcion_consulta','tipo_consulta','nombre_vecino','descripcion_vecino',
    'tipo_vecino','pps_evaluador_2','pps_arbitraje']],'md4_segunda_ronda_si_corresponde.csv')
save_csv(PPS.merge(pairs[['consulta','vecino','id_par']],on=['consulta','vecino'],how='left'),
    'md4_posiciones_pps.csv')
# Ten unrelated adjustment queries; these are never selected from retained Q.
calib_pool=[]
for qi in calibration:
    candidates=base['ranks'][int(qi)]
    if candidates:
        j=candidates[0][0]
        calib_pool.append({'consulta':SRC.article_id.iloc[qi],
             'vecino':SRC.article_id.iloc[j],
             'nombre_consulta':SRC.prod_name.iloc[qi],
             'nombre_vecino':SRC.prod_name.iloc[j]})
    if len(calib_pool)>=10:break
save_csv(calib_pool,'md4_recalibracion_10_pares_ajuste.csv')
save_json({'U_pares_unicos_reales':U,'n_primera_ronda':n_second,
    'porcentaje_efectivo':100*n_second/U if U else 0,
    'n_segunda_ronda_disjunta':len(second_indices),'D_pps_por_modelo':150,
    'estado':'PENDIENTE_CALIFICACION_HUMANA_REAL'},'md4_plan_revision_humana.json')

# Optional scored file: export md4_plantilla_pps_ciega.csv, fill genuine first/second
# reviewer scores, and reattach as md4_pps_evaluaciones.csv via Kaggle Add Input.
paths=[]
for root in [Path('/kaggle/input'),OUT,Path.cwd()]:
    if root.exists():paths.extend(root.rglob('md4_pps_evaluaciones.csv'))
paths=list(dict.fromkeys(p.resolve() for p in paths))
if len(paths)>1:raise ValueError('Más de un archivo de respuestas PPS: '+str(paths))
REVISION_REAL=bool(paths)
def valid_score(s):return pd.to_numeric(s,errors='coerce')
if REVISION_REAL:
    rated=pd.read_csv(paths[0],dtype={'id_par':str,'consulta':str,'vecino':str})
    assert {'id_par','consulta','vecino','pps_evaluador_1','pps_evaluador_2'}<=set(rated)
    assert rated.id_par.is_unique
    assert set(rated.id_par).issubset(set(allpairs.id_par))
    ratings=pairs[['id_par','consulta','vecino','seleccion_primera_ronda_30pct',
        'seleccion_segunda_ronda_si_kappa_menor_060']].merge(
        rated[['id_par','consulta','vecino','pps_evaluador_1','pps_evaluador_2']+
              (['pps_arbitraje'] if 'pps_arbitraje' in rated else [])],on=['id_par','consulta','vecino'],how='left')
    for col in ['pps_evaluador_1','pps_evaluador_2']:
        _raw=ratings[col].copy()
        ratings[col]=valid_score(_raw)
        assert not (_raw.notna() & _raw.astype(str).str.strip().ne('') & ratings[col].isna()).any(), f'{col}: texto no numérico'
        assert ratings[col].dropna().isin([0,1,2]).all(),col
    if 'pps_arbitraje' in ratings:
        _raw_arb=ratings['pps_arbitraje'].copy()
        ratings['pps_arbitraje']=valid_score(_raw_arb)
        assert not (_raw_arb.notna() & _raw_arb.astype(str).str.strip().ne('') & ratings['pps_arbitraje'].isna()).any(), 'pps_arbitraje: texto no numérico'
        assert ratings['pps_arbitraje'].dropna().isin([0,1,2]).all(), 'pps_arbitraje'
    else:ratings['pps_arbitraje']=np.nan
    same_tipo=ratings.consulta.map(lookup['product_type_name']).eq(
        ratings.vecino.map(lookup['product_type_name']))
    for _score_col in ('pps_evaluador_1','pps_evaluador_2','pps_arbitraje'):
        _scores=ratings[_score_col]
        _bad_domain=((_scores==1)&~same_tipo)|((_scores==2)&same_tipo)
        assert not _bad_domain.any(), f'{_score_col}: incoherencia con product_type_name'
    ratings['pps_final']=ratings['pps_evaluador_1'].copy()
    conflicted=(ratings.pps_evaluador_1.notna() & ratings.pps_evaluador_2.notna() &
                (ratings.pps_evaluador_1!=ratings.pps_evaluador_2))
    decided=conflicted & ratings.pps_arbitraje.notna()
    ratings.loc[decided,'pps_final']=ratings.loc[decided,'pps_arbitraje']
    # Rubric: intra-type level 1, inter-type level 2. No fake scores.
    same=(ratings.consulta.map(lookup['product_type_name'])==
          ratings.vecino.map(lookup['product_type_name']))
    bad=((ratings.pps_evaluador_1==1)&(~same))|((ratings.pps_evaluador_1==2)&same)
    assert not bad.any(), 'PPS1/2 inconsistente con product_type_name'
    reviewed_first=ratings[ratings.seleccion_primera_ronda_30pct]
    if reviewed_first[['pps_evaluador_1','pps_evaluador_2']].notna().all(axis=1).all():
        kappa=float(cohen_kappa_score(reviewed_first.pps_evaluador_1,
                                    reviewed_first.pps_evaluador_2))
        agreement=float(np.mean(reviewed_first.pps_evaluador_1==reviewed_first.pps_evaluador_2))
        concord=[{'ronda':1,'n':len(reviewed_first),'porcentaje_real_U':100*len(reviewed_first)/U,
                  'kappa':kappa,'acuerdo':agreement,'estado':'EVALUADA'}]
    else:
        kappa=np.nan
        concord=[{'ronda':1,'n':len(reviewed_first),'kappa':np.nan,
                  'acuerdo':np.nan,'estado':'PENDIENTE'}]
    resolucion_kappa=True
    if np.isfinite(kappa) and kappa<.60:
        reviewed_second=ratings[ratings.seleccion_segunda_ronda_si_kappa_menor_060]
        if reviewed_second[['pps_evaluador_1','pps_evaluador_2']].notna().all(axis=1).all():
            k2=float(cohen_kappa_score(reviewed_second.pps_evaluador_1,reviewed_second.pps_evaluador_2))
            a2=float(np.mean(reviewed_second.pps_evaluador_1==reviewed_second.pps_evaluador_2))
            disagreement=reviewed_second[reviewed_second.pps_evaluador_1!=reviewed_second.pps_evaluador_2]
            arbitrated=disagreement.pps_arbitraje.notna().all()
            # There must also be independent documentary proof of the actual 10-pair recalibration.
            evidence_candidates=[p for root in (OUT,Path('/kaggle/input')) if root.exists()
                                 for p in root.rglob('md4_recalibracion_real.json')]
            evidence_candidates=list(dict.fromkeys(p.resolve() for p in evidence_candidates))
            if len(evidence_candidates)>1:raise RuntimeError('Más de una evidencia de recalibración')
            calibration_done=False
            if evidence_candidates:
                evidence=json.loads(evidence_candidates[0].read_text(encoding='utf-8'))
                calibration_done=(evidence.get('realizada') is True and
                                  evidence.get('pares_revisados')==10 and
                                  bool(evidence.get('fecha')))
            resolucion_kappa=bool(arbitrated and calibration_done and np.isfinite(k2))
            concord.append({'ronda':2,'n':len(reviewed_second),'kappa':k2,'acuerdo':a2,
                'estado':'EVALUADA' if resolucion_kappa else 'PENDIENTE_RECALIBRACION_O_ARBITRAJE',
                'requiere_arbitraje':len(disagreement)>0,'arbitrajes_resueltos':int(disagreement.pps_arbitraje.notna().sum()),
                'recalibracion_10_realizada':calibration_done})
        else:
            resolucion_kappa=False
            concord.append({'ronda':2,'n':len(reviewed_second),'estado':'PENDIENTE_RECALIBRACION_Y_REVISION'})
    save_csv(concord,'md4_concordancia_pps.csv')
    # Model scores only after every genuine unique pair has been rated.
    all_first=ratings.pps_evaluador_1.notna().all()
    if all_first and (np.isfinite(kappa) and
                     (kappa>=.60 or resolucion_kappa)):
        ps=ratings[['id_par','pps_final']].rename(columns={'pps_final':'pps_evaluador_1'})
        quantitative=PPS.merge(pairs[['consulta','vecino','id_par']],on=['consulta','vecino'],how='left').merge(ps,on='id_par',how='left')
        model_scores=[]
        for mod in ('TFIDF','LSA','SBERT'):
            x=quantitative[quantitative.modelo==mod]
            assert len(x)<=150 and x.pps_evaluador_1.notna().all()
            c=x.pps_evaluador_1.value_counts()
            model_scores.append({'modelo':mod,'D':150,'n0':int(c.get(0,0)+150-len(x)),
                'n1':int(c.get(1,0)),'n2':int(c.get(2,0)),
                'P_PPS0':float((c.get(0,0)+150-len(x))/150),
                'P_PPS1':float(c.get(1,0)/150),'P_PPS2':float(c.get(2,0)/150)})
        save_csv(model_scores,'md4_pps_resultados_reales.csv')
        reference=next(z for z in model_scores if z['modelo']=='TFIDF')
        verdicts=[]
        for x in model_scores:
            if x['modelo']=='TFIDF':continue
            improved=x['n2']-reference['n2']>=15 and x['n0']-reference['n0']<=7
            verdicts.append({'modelo':x['modelo'],'supera_baseline_en_completo':improved,
                'delta_PPS2':x['P_PPS2']-reference['P_PPS2'],
                'delta_PPS0':x['P_PPS0']-reference['P_PPS0'],
                'decision_final':'PENDIENTE_RECALIFICAR_VARIANTES_IMPUTACION'})
        
        save_csv(verdicts,'md4_evaluacion_criterio_pps.csv')
        # Genuine ratings reused only when the SAME pair occurs again; unseen
        # neighbors must be explicitly reviewed before a sensitivity conclusion.
        rated_final=rated[['id_par','pps_evaluador_1']+
                          (['pps_arbitraje'] if 'pps_arbitraje' in rated else [])].copy()
        rated_final['pps_evaluador_1']=pd.to_numeric(rated_final.pps_evaluador_1,errors='coerce')
        if 'pps_arbitraje' in rated_final:
            arb=pd.to_numeric(rated_final.pps_arbitraje,errors='coerce')
            rated_final.loc[arb.notna(),'pps_evaluador_1']=arb[arb.notna()].to_numpy()
        all_rated=allpairs[['id_par','consulta','vecino']].merge(
            rated_final[['id_par','pps_evaluador_1']],on='id_par',how='left')
        by_reg=[]
        for regime in ('completo','sin_imputados_modelo_fijo','refit_orig'):
            for model in ('TFIDF','LSA','SBERT'):
                if regime=='refit_orig' and model=='SBERT':continue
                rr=ranking_df[(ranking_df.regimen==regime)&(ranking_df.modelo==model)&(ranking_df.k==5)]
                m=rr.merge(all_rated[['consulta','vecino','pps_evaluador_1']],on=['consulta','vecino'],how='left')
                available=m.pps_evaluador_1.notna().all()
                counts=m.pps_evaluador_1.value_counts() if available else pd.Series(dtype=int)
                by_reg.append({'regimen':regime,'modelo':model,'D':150,'n_pares':len(m),
                    'n_pares_sin_calificar':int(m.pps_evaluador_1.isna().sum()),
                    'PPS0':(int(counts.get(0,0))+150-len(m))/150 if available else np.nan,
                    'PPS1':int(counts.get(1,0))/150 if available else np.nan,
                    'PPS2':int(counts.get(2,0))/150 if available else np.nan,
                    'estado':'COMPLETO' if available else 'REQUIERE_REVISION'})
        save_csv(by_reg,'md4_pps_por_regimen.csv')
        checks_by_reg=[]
        for regime in ('completo','sin_imputados_modelo_fijo','refit_orig'):
            rmap={x['modelo']:x for x in by_reg if x['regimen']==regime}
            baseline=rmap['TFIDF']
            for model in ('LSA','SBERT'):
                if model not in rmap:continue
                alt=rmap[model]
                valid=alt['estado']=='COMPLETO' and baseline['estado']=='COMPLETO'
                verdict=(alt['PPS2']-baseline['PPS2']>=.10-1e-12 and
                         alt['PPS0']-baseline['PPS0']<=.05+1e-12) if valid else None
                checks_by_reg.append({'regimen':regime,'modelo':model,'evaluable':valid,
                    'supera_baseline':verdict})
        save_csv(checks_by_reg,'md4_criterio_pps_por_regimen.csv')
        complete_final=all(x['evaluable'] for x in checks_by_reg)
        if complete_final:
            summary=[]
            for model in ('LSA','SBERT'):
                # For pre-trained SBERT, refit_orig is the same as filtered.
                checks=[x for x in checks_by_reg if x['modelo']==model]
                required=['completo','sin_imputados_modelo_fijo'] + (['refit_orig'] if model=='LSA' else [])
                summary.append({'modelo':model,'supera_en_todos_los_regimenes':
                    all(x['supera_baseline'] for x in checks if x['regimen'] in required)})
            save_csv(summary,'md4_decision_final_condicional.csv')

        registro('MD4-P02','PPS_Kappa','PASS_CON_RESERVA',
            'Evaluación humana principal recibida; falta calificar pares nuevos de escenarios sin imputados/refit')
    else:
        registro('MD4-P02','PPS_Kappa','PENDIENTE',
            'Calificaciones incompletas, concordancia pendiente o recalibración pendiente')
else:
    save_csv([{'ronda':1,'n':n_second,'kappa':np.nan,'acuerdo':np.nan,'estado':'PENDIENTE_REVISION_HUMANA'}],
             'md4_concordancia_pps.csv')
    registro('MD4-P02','PPS_Kappa','PENDIENTE_REVISION_HUMANA',
             f'U={U}; primera revisión doble = {n_second}, 30% efectivo={100*n_second/U:.3f}')

Guardado: /kaggle/working/hm_modelamiento/md4/md4_plantilla_pps_todos_escenarios.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_plantilla_pps_ciega.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_primera_ronda_segundo_evaluador.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_segunda_ronda_si_corresponde.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_posiciones_pps.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_recalibracion_10_pares_ajuste.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_plan_revision_humana.json
Guardado: /kaggle/working/hm_modelamiento/md4/md4_concordancia_pps.csv


## 9. MD4-P04 sensibilidad de corpus, errores y limitaciones
Corpus enriquecido incorpora explícitamente clasificación del catálogo: analizar inflación taxonómica. Comparar ajuste completo, originales y únicos; revisar E1 imputados, E2 textos ultracortos y E3 Unknown. No confundir un vecino de otro tipo con pertinencia humana.

In [20]:
# MD4-P04 sensitivity, using actual top-k results, not just fitting model counts.
SENSITIVITY=[]
def jac_rank(A,B,idx,k=10):
    vals=[]
    for q in idx:
        a={j for j,_ in A.get(int(q),[])[:k]}; b={j for j,_ in B.get(int(q),[])[:k]}
        vals.append(float(len(a&b)/len(a|b)) if (a or b) else np.nan)
    return float(np.nanmean(vals)) if np.isfinite(vals).any() else np.nan

def structural_of_ranks(ranks,queries,k=10):
    D=k*len(queries);n=0;macro=0;typ=0
    for q in queries:
        for j,_ in ranks.get(int(q),[])[:k]:
            n+=1; macro+=int(SRC.product_group_name.iloc[q]==SRC.product_group_name.iloc[j])
            typ+=int(SRC.tipo.iloc[q]==SRC.tipo.iloc[j])
    return {'TCC':n/D if D else np.nan,'TCMG_propuesta':macro/D if D else np.nan,
            'TCTipo_propuesta':typ/D if D else np.nan,'vacantes':D-n,'D':D}

# Fit against deduplicated text (unweighted) and compare with full document-frequency fit.
vector_u,mat_u=tfidf_fit(unique_texts,base['sw'],base['ng'])
mat_u_full=vector_u.transform(texts).tocsr()
svd_u=TruncatedSVD(n_components=CHOSEN_LSA,random_state=20261011)
svd_u.fit(mat_u)
lsa_u_full=normalize(svd_u.transform(mat_u_full).astype('float32'))
# Weighted unique strings represent X.T@X of FULL TF-IDF with frozen full vectorizer:
# each normalized unique row occurs frequency times; multiply its row by sqrt(freq).
counts_u=np.bincount(inverse,minlength=len(unique_texts))
mat_unique_with_full_idf=base['vec'].transform(unique_texts)
mat_w=mat_unique_with_full_idf.multiply(np.sqrt(counts_u)[:,None]).tocsr()
svd_w=TruncatedSVD(n_components=CHOSEN_LSA,random_state=20261011)
svd_w.fit(mat_w)
lsa_w_full=normalize(svd_w.transform(base['mat']).astype('float32'))
# Independent algebraic check on random vectors, without allocating X.T@X.
rng_a=np.random.default_rng(20261011)
columns=rng_a.choice(base['mat'].shape[1],size=min(32,base['mat'].shape[1]),replace=False)
for j in columns:
    left=float(base['mat'][:,j].power(2).sum())
    right=float(mat_w[:,j].power(2).sum())
    assert np.isclose(left,right,rtol=1e-5,atol=1e-5),(j,left,right)

ref_tfidf=base['ranks'];ref_lsa=LSA_RESULTS[CHOSEN_LSA]['ranks']
CANDIDATES=[('originales_sin_imputados_LSA','LSA',lsa_full),
            ('originales_sin_imputados_TF','TFIDF',mat_full),
            ('textos_unicos_no_ponderados_TF','TFIDF',mat_u_full),
            ('textos_unicos_no_ponderados_LSA','LSA',lsa_u_full),
            ('textos_unicos_ponderados_LSA','LSA',lsa_w_full)]
# The alternate corpus contains explicit taxonomy terms; any apparent gain may be leakage.
enriched=SRC.corpus_texto_nlp.fillna('').astype(str).to_numpy()
v_en,tf_en=tfidf_fit(enriched,base['sw'],base['ng'])
CANDIDATES.append(('corpus_enriquecido_TF','TFIDF',tf_en))
ls_en=TruncatedSVD(n_components=CHOSEN_LSA,random_state=20261011)
lsa_en=normalize(ls_en.fit_transform(tf_en).astype('float32'))
CANDIDATES.append(('corpus_enriquecido_LSA','LSA',lsa_en))
# Sensitivity of n-grams and stopwords using all four TF-IDF alternatives and LSA on their exact CSR.
for label,item in TF_RESULTS.items():
    if label!=CHOSEN_TF:
        CANDIDATES.append(('hiperparametros_'+label,'TFIDF',item['mat']))
        alt_svd=TruncatedSVD(n_components=CHOSEN_LSA,random_state=20261011)
        lsa_alt=normalize(alt_svd.fit_transform(item['mat']).astype('float32'))
        CANDIDATES.append(('hiperparametros_'+label,'LSA',lsa_alt))

for label,model,mat in CANDIDATES:
    rankings=batch_ranking(calibration,mat,top=10)
    reference=ref_tfidf if model=='TFIDF' else ref_lsa
    for k in (5,10):
        metr=structural_of_ranks(rankings,calibration,k)
        main=structural_of_ranks(reference,calibration,k)
        SENSITIVITY.append({'ensayo':label,'modelo':model,'estrato':'M_ajuste',
           'n_consultas':len(calibration),'k':k,'jaccard_vs_principal':jac_rank(reference,rankings,calibration,k),
           **metr,'delta_TCMG':metr['TCMG_propuesta']-main['TCMG_propuesta'],
           'delta_TCTipo':metr['TCTipo_propuesta']-main['TCTipo_propuesta']})
    save_csv(SENSITIVITY,'md4_sensibilidad_corpus.csv')
# E1/E2/E3 measured on ALL indices; these strata may overlap, and report their sizes.
case_indices={'E1_imputados':np.flatnonzero(SRC.imputado.to_numpy()),
              'E2_texto_menos_3_palabras':np.flatnonzero((SRC.n_palabras<3).to_numpy()),
              'E3_Unknown':np.flatnonzero((SRC.tipo=='Unknown').to_numpy())}
for label,idx in case_indices.items():
    for model,mat in MODEL_REPS.items():
        rar=batch_ranking(idx,mat,top=10)
        for k in (5,10):
            metr=structural_of_ranks(rar,idx,k)
            SENSITIVITY.append({'ensayo':'caso_especial','modelo':model,'estrato':label,
                'n_consultas':len(idx),'k':k,'jaccard_vs_principal':np.nan,**metr})
    save_csv(SENSITIVITY,'md4_sensibilidad_corpus.csv')
# Text-equality exclusion sensitivity uses same model weights, same held queries.
for model,mat in MODEL_REPS.items():
    with_filter=SEARCH_RESULTS[(model,'completo')]
    no_filter={int(q):rank_one(int(q),mat,top=10,exclude_identical=False) for q in held}
    for k in (5,10):
        a=structural_of_ranks(with_filter,held,k);b=structural_of_ranks(no_filter,held,k)
        SENSITIVITY.append({'ensayo':'sin_filtro_texto_igual','modelo':model,'estrato':'Q_retenido',
            'n_consultas':len(held),'k':k,'jaccard_vs_principal':jac_rank(with_filter,no_filter,held,k),
            **b,'delta_TCMG':b['TCMG_propuesta']-a['TCMG_propuesta'],
            'delta_TCTipo':b['TCTipo_propuesta']-a['TCTipo_propuesta']})
save_csv(SENSITIVITY,'md4_sensibilidad_corpus.csv')

# Prueba algebraica de Gram: comprobación de productos CRUZADOS (no solo diagonal).
_rng=np.random.default_rng(20261011)
GRAM_TESTS=[]
for _i in range(6):
    _a=np.zeros(base['mat'].shape[1],dtype=np.float64)
    _b=np.zeros_like(_a)
    _ia=_rng.choice(len(_a),min(48,len(_a)),replace=False)
    _ib=_rng.choice(len(_a),min(48,len(_a)),replace=False)
    _a[_ia]=_rng.normal(size=len(_ia));_b[_ib]=_rng.normal(size=len(_ib))
    _b[_ia[0]]=1.0
    _v1=float(np.dot(np.asarray(base['mat']@_a),np.asarray(base['mat']@_b)))
    _v2=float(np.dot(np.asarray(mat_w@_a),np.asarray(mat_w@_b)))
    assert np.isclose(_v1,_v2,rtol=2e-4,atol=1e-4),(_v1,_v2)
    GRAM_TESTS.append({'prueba':_i,'producto_full':_v1,'producto_ponderado':_v2,'abs_error':abs(_v1-_v2)})
save_csv(GRAM_TESTS,'md4_verificacion_gram_bilineal.csv')

save_json({'gram_equivalente_verificado':True,
    'filas_texto_unico':len(unique_texts), 'filas_catalogo':len(SRC),
    'imputados':int(SRC.imputado.sum()), 'strata_observados':{k:len(v) for k,v in case_indices.items()},
    'definiciones_tasas_sujetas_a_ratificacion':True},'md4_sensibilidad_resumen.json')
registro('MD4-P04','ranking_sensibilidad','PASS_CON_RESERVA',
    'Todas las matrices y estratos evaluados; fórmulas estructurales provisionales')

Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamiento/md4/md4_sensibilidad_corpus.csv
Guardado: /kaggle/working/hm_modelamie

{'prueba': 'MD4-P04',
 'etapa': 'ranking_sensibilidad',
 'estado': 'PASS_CON_RESERVA',
 'detalle': 'Todas las matrices y estratos evaluados; fórmulas estructurales provisionales',
 'duracion_s': None}

## 10. Conclusiones y exportación
**No existe ganador MD-4 hasta completar PPS y Kappa.** Regla documental: sobre 150 posiciones por modelo, alternativa mejora PPS=2 en al menos 15 posiciones y no aumenta PPS=0 en más de 7; debe sostenerse en los regímenes de imputación. Conservar `md4_hoja_revisores_pps.csv`, índices, rankings, métricas y versiones.

## Evaluación de los modelos — Semana 12 (ejecutada en este mismo notebook)
Esta sección **calcula e interpreta la evaluación con las predicciones, clusters, reglas o rankings reales** generados previamente. Incluye controles de cobertura P01–P04, coherencia numérica, tablas y gráficos para el informe. Si el entrenamiento fue interrumpido, informa `EVALUACION_INCOMPLETA`: no produce resultados inventados. Los archivos generados se incluirán automáticamente en el ZIP final.


In [21]:
# SEMANA 12 — evaluación MD-4 dentro del mismo notebook y sin inventar PPS.
# Modelado y evaluación automática ya ejecutados arriba; el juicio humano es condicional.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
W12=Path(OUT)
def w12_csv(name):
    p=W12/name
    return pd.read_csv(p) if p.is_file() and p.stat().st_size else pd.DataFrame()
def w12_json(name):
    p=W12/name
    return json.loads(p.read_text(encoding='utf-8')) if p.is_file() else None

def w12_plot(name):
    plt.tight_layout();plt.savefig(W12/name,dpi=145,bbox_inches='tight');plt.close()

corpus=w12_csv('md4_auditoria_corpus.csv')
params=w12_csv('md4_seleccion_hiperparametros.csv')
tfidf=w12_csv('md4_seleccion_tfidf.csv')
lsa=w12_csv('md4_seleccion_lsa.csv')
rank=w12_csv('md4_comparacion_topk_modelos.csv')
struct=w12_csv('md4_aporte_vs_jerarquia.csv')
overlap=w12_csv('md4_solapamiento_modelos.csv')
sensitivity=w12_csv('md4_sensibilidad_corpus.csv')
concord=w12_csv('md4_concordancia_pps.csv')
pps=w12_csv('md4_pps_resultados_reales.csv')
by_reg=w12_csv('md4_criterio_pps_por_regimen.csv')
plan=w12_json('md4_plan_revision_humana.json')
checks=[]
def w12_check(prueba,componente,ok,evidence):
    checks.append({'prueba':prueba,'componente':componente,'estado':'PASS' if ok else 'PENDIENTE',
                   'evidencia':evidence})
p01={'campo','valor'}<=set(corpus) and {'N_articulos','N_descripciones_imputadas','max_wordpieces_unicos'}<=set(corpus.campo)
w12_check('MD4-P01','corpus y truncamiento WordPieces',p01,'md4_auditoria_corpus.csv; md4_tokenizador.json')
p02a={'modelo','tcc10','entropia'}<=set(tfidf) and len(tfidf)==4 and \
      {'dimensiones','varianza','jaccard'}<=set(lsa) and {100,200}<=set(lsa.dimensiones)
w12_check('MD4-P02','cuatro TF-IDF y dos LSA evaluados en ajuste',p02a,
          'md4_seleccion_tfidf.csv; md4_seleccion_lsa.csv')
w12_check('MD4-P02','hiperparámetros consolidados y congelados',len(params)==6,
          'md4_seleccion_hiperparametros.csv')
p02b={'modelo','regimen','k','consulta','vecino','coseno'}<=set(rank) and \
      {'TFIDF','LSA','SBERT'}<=set(rank.modelo.astype(str)) and \
      {'completo','sin_imputados_modelo_fijo','refit_orig'}<=set(rank.regimen.astype(str))
w12_check('MD4-P02','rankings top-5/10 y variantes',p02b,'md4_comparacion_topk_modelos.csv')
# El documento aún no define de manera inequívoca las fórmulas de todos los acrónimos.
p03={'modelo','regimen','k','recuperados','D'}<=set(struct) and not struct.empty
w12_check('MD4-P03','conteos estructurales auditables',p03,
          'md4_aporte_vs_jerarquia.csv; definiciones provisionales por ratificar')
p04={'ensayo','modelo','estrato','jaccard_vs_principal'}<=set(sensitivity) and not sensitivity.empty and \
     {'E1_imputados','E2_texto_menos_3_palabras','E3_Unknown','Q_retenido','M_ajuste'}<=set(sensitivity.estrato.astype(str)) and \
     {'originales_sin_imputados_LSA','originales_sin_imputados_TF'}<=set(sensitivity.ensayo.astype(str))
# La sensibilidad adicional puede existir en la revisión complementaria, sin alterar los CSV históricos.
revision=W12/'revision_complementaria'/'md4_sensibilidad_corpus_consolidada.csv'
if not p04 and revision.is_file():
    consolidated=pd.read_csv(revision)
    p04={'originales_sin_imputados_LSA','originales_sin_imputados_TF'}<=set(consolidated.ensayo.astype(str)) and \
      {'E1_imputados','E2_texto_menos_3_palabras','E3_Unknown','Q_retenido','M_ajuste'}<=set(consolidated.estrato.astype(str))
w12_check('MD4-P04','sensibilidad y estratos especiales completos',p04,
          'md4_sensibilidad_corpus.csv; revisión complementaria si existe')

_second_done=({'ronda','estado','recalibracion_10_realizada'}<=set(concord) and
       not concord[(concord.ronda==2)&(concord.estado.astype(str)=='EVALUADA')&
          (concord.recalibracion_10_realizada.astype(str).str.lower()=='true')].empty)
human_agreement=({'kappa','estado'}<=set(concord) and concord.kappa.notna().any() and
       (concord.loc[concord.kappa.notna(),'kappa'].ge(.60).any() or _second_done))
human_done=(plan is not None and {'modelo','n0','n1','n2'}<=set(pps) and len(pps)==3 and
            {'kappa','acuerdo','estado'}<=set(concord) and human_agreement and
            {'regimen','modelo','evaluable','supera_baseline'}<=set(by_reg) and
            by_reg.evaluable.astype(str).str.lower().eq('true').all())
w12_check('MD4-P02/P03','PPS reales, dos evaluadores, Kappa y robustez de mejora',human_done,
          'md4_pps_resultados_reales.csv; md4_concordancia_pps.csv; md4_criterio_pps_por_regimen.csv')

invariants=[]
if p02b:
    unique=not rank.duplicated(['modelo','regimen','k','consulta','vecino']).any()
    invariants.append({'control':'no_duplicados_rank','estado':'PASS' if unique else 'FALLO'})
    groups=rank.groupby(['modelo','regimen','k','consulta']).rango.max() if 'rango' in rank else pd.Series(dtype=float)
    invariants.append({'control':'rango_en_topk','estado':'PASS' if \
       (not groups.empty and all(1<=int(r)<=int(k) for (_,_,k,_),r in groups.items())) else 'FALLO'})
if pps is not None and not pps.empty and {'n0','n1','n2','D'}<=set(pps):
    valid=bool(((pps.n0+pps.n1+pps.n2)==pps.D).all())
    invariants.append({'control':'PPS_n0+n1+n2_equals_D','estado':'PASS' if valid else 'FALLO'})
internal_ok=all(i['estado']=='PASS' for i in invariants)
if not internal_ok:
    for x in checks:x['estado']='REVISAR_INVARIANTES'
pd.DataFrame(checks).to_csv(W12/'md4_cobertura_semana12.csv',index=False)
pd.DataFrame(invariants,columns=['control','estado']).to_csv(W12/'md4_invariantes_evaluacion.csv',index=False)

# Figuras: nunca interpretar coseno absoluto entre espacios diferentes como ganancia de calidad.
if p02b:
    primary=rank[(rank.regimen=='completo')&(rank.k==5)].copy()
    if not primary.empty:
        fig,ax=plt.subplots(figsize=(9,4.5))
        for name,g in primary.groupby('modelo'):
            ax.hist(pd.to_numeric(g.coseno,errors='coerce'),bins=24,alpha=.4,label=name)
        ax.set(title='Distribución de cosenos top-5 (no métrica de pertinencia comparable)',xlabel='Coseno',ylabel='Recuento')
        ax.legend();w12_plot('md4_evaluacion_cosenos.png')
if {'regimen','modelo_A','modelo_B','k','jaccard_media'}<=set(overlap):
    fig,ax=plt.subplots(figsize=(9,4.5))
    complete=overlap[overlap.regimen=='completo']
    for (aa,bb),g in complete.groupby(['modelo_A','modelo_B']):
        ax.plot(g.k,g.jaccard_media,marker='o',label=aa+' / '+bb)
    ax.set(xlabel='Top-k',ylabel='Jaccard medio',ylim=(0,1),title='Solapamiento de rankings en holdout')
    ax.legend();w12_plot('md4_evaluacion_solapamiento.png')
if {'modelo','regimen','k','recuperados','D','TCMG_at_k_PROPUESTA','TCTipo_at_k_PROPUESTA'}<=set(struct):
    g=struct[(struct.regimen=='completo')&(struct.k==5)]
    if not g.empty:
        fig,ax=plt.subplots(figsize=(8.5,4.5))
        xx=np.arange(len(g));ax.bar(xx-.2,g.TCMG_at_k_PROPUESTA,width=.4,label='Mismo macrogrupo')
        ax.bar(xx+.2,g.TCTipo_at_k_PROPUESTA,width=.4,label='Mismo tipo')
        ax.set_xticks(xx,g.modelo);ax.set(ylabel='Proporción estructural (fórmula provisional)',title='Jerarquía existente: conteos descriptivos')
        ax.legend();w12_plot('md4_evaluacion_jerarquia.png')
if {'modelo','P_PPS2','P_PPS0'}<=set(pps) and human_done:
    fig,ax=plt.subplots(figsize=(8,4))
    xx=np.arange(len(pps));ax.bar(xx-.2,pps.P_PPS2,width=.4,label='PPS 2')
    ax.bar(xx+.2,pps.P_PPS0,width=.4,label='PPS 0')
    ax.set_xticks(xx,pps.modelo);ax.set(ylabel='Fracción de 150 posiciones',title='Calificación PPS humana real')
    ax.legend();w12_plot('md4_evaluacion_pps_humana.png')

lines=['# MD-4 — evaluación Semana 12','',
 'Se modelaron representaciones TF-IDF, LSA y Sentence-BERT y se evaluaron rankings retenidos, cobertura y sensibilidad.',
 'Las proporciones de pertenencia a categoría son controles estructurales, no sustituyen la pertinencia humana.',
 '', '## Cobertura']
lines += [f"- **{x['prueba']} / {x['componente']}**: {x['estado']} (`{x['evidencia']}`)." for x in checks]
lines += ['', '## Comparación automática']
if p02b and p03:
    for model,g in struct[(struct.regimen=='completo')&(struct.k==5)].groupby('modelo'):
        z=g.iloc[0];lines.append(f'- {model}: cobertura top-5={z.recuperados}/{z.D}; mismo tipo={z.TCTipo_at_k_PROPUESTA:.3f} (**definición propuesta**).')
if human_done:
    for z in pps.itertuples():lines.append(f'- {z.modelo}: PPS2={int(z.n2)}/150; PPS0={int(z.n0)}/150.')
else:lines.append('- **No hay conclusión de superioridad semántica humana**: falta calificación PPS real, Kappa o evaluación de variantes.')
if not internal_ok:lines.append('- **ALERTA:** alguna invariante de métricas falló.')
lines += ['', '## Restricciones',
 '- Nunca se asignan puntajes humanos ficticios ni se declara un modelo ganador sin PPS.',
 '- No usar holdout para ajustar ngramas, stopwords, dimensión LSA o seleccionar modelos.',
 '- Fórmulas TCT, TCMG, TDIG, TCTipo y TDIT deben ratificarse antes de presentarlas como indicadores aprobados.',
 '', '## Evidencias',
 '- `md4_cobertura_semana12.csv`, `md4_invariantes_evaluacion.csv`, gráficos y rankings/controles previos.',
 '- La hoja PPS puede completarse y reimportarse en una nueva ejecución del mismo notebook; las puntuaciones deben ser reales.']
(W12/'md4_informe_evaluacion_semana12.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
status='EVALUADO' if all(x['estado']=='PASS' for x in checks) and internal_ok else 'EVALUACION_INCOMPLETA'
(W12/'md4_estado_semana12.json').write_text(json.dumps({'estado':status,'pruebas':checks,
    'invariantes_aprobadas':internal_ok,'evaluacion_humana_real':human_done,
    'resultados_verdaderos_no_simulados':True},ensure_ascii=False,indent=2),encoding='utf-8')
print('EVALUACIÓN SEMANA 12 — MD-4:',status)
print(pd.DataFrame(checks).to_string(index=False))


EVALUACIÓN SEMANA 12 — MD-4: EVALUACION_INCOMPLETA
     prueba                                              componente    estado                                                                                 evidencia
    MD4-P01                        corpus y truncamiento WordPieces      PASS                                            md4_auditoria_corpus.csv; md4_tokenizador.json
    MD4-P02             cuatro TF-IDF y dos LSA evaluados en ajuste      PASS                                            md4_seleccion_tfidf.csv; md4_seleccion_lsa.csv
    MD4-P02               hiperparámetros consolidados y congelados      PASS                                                         md4_seleccion_hiperparametros.csv
    MD4-P02                           rankings top-5/10 y variantes      PASS                                                          md4_comparacion_topk_modelos.csv
    MD4-P03                        conteos estructurales auditables      PASS                     md4_aporte_

In [22]:

ARTIFACTS_MD4=['md4_auditoria_corpus.csv','md4_seleccion_hiperparametros.csv',
               'md4_comparacion_topk_modelos.csv','md4_aporte_vs_jerarquia.csv',
               'md4_concordancia_pps.csv','md4_sensibilidad_corpus.csv',
               'md4_plantilla_pps_ciega.csv','md4_plantilla_pps_todos_escenarios.csv','md4_solapamiento_modelos.csv']
missing=[x for x in ARTIFACTS_MD4 if not (OUT/x).exists()]
completed=not missing and (OUT/'md4_pps_resultados_reales.csv').exists()
final_status='PARCIAL_HUMANO_PENDIENTE' if not completed else 'REVISAR_ANTECEDENTES_PPS_VARIANTES'
save_json({'faltantes':missing,'evaluacion_humana_real':(OUT/'md4_pps_resultados_reales.csv').exists(),
 'definiciones_acronimos_pendientes':True,'estado':final_status},'md4_auditoria_entregables.json')
print('MD4:',final_status,'faltantes:',missing)


Guardado: /kaggle/working/hm_modelamiento/md4/md4_auditoria_entregables.json
MD4: PARCIAL_HUMANO_PENDIENTE faltantes: []


## Exportar evidencia compacta Semana 12 a ZIP
El paquete público omite ID de artículos asociados a consultas, descripciones, fingerprints de fuentes y embeddings. Para completar PPS, conserva aparte las plantillas privadas en Kaggle.

In [23]:
# CELDA FINAL — exportar evidencia MD-1 sin volver a ejecutar modelos.
# Permite descargar incluso una ejecución parcial si quedaron artefactos.
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, FileLink
import hashlib, json, zipfile

OBJETIVO = 'md4'

def exportar_evidencias(objetivo, carpeta=None, zip_destino=None):
    if carpeta is None:
        carpeta = Path('/kaggle/working/hm_modelamiento') / objetivo
    carpeta = Path(carpeta).resolve()
    if not carpeta.is_dir():
        raise FileNotFoundError(f'No existe la carpeta de resultados: {carpeta}')
    if zip_destino is None:
        base = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else carpeta.parent
        zip_destino = base / f'HM_{objetivo.upper()}_Evidencias.zip'
    zip_destino = Path(zip_destino).resolve()
    zip_destino.parent.mkdir(parents=True, exist_ok=True)

    extensiones = {'.csv', '.json', '.jsonl', '.txt', '.md', '.png', '.jpg', '.jpeg', '.svg', '.html'}
    directorios_no_publicables = {'incidencia', 'cache_ajustes', '__pycache__', 'hf_cache'}
    archivos_excluidos = {'entrada_y_entorno.json', 'origen_cache.json',
        'referencia_seleccion.csv', 'referencia_test.csv',
        'md4_consultas_holdout.csv','md4_consultas_ajuste.csv',
        'md4_plantilla_pps_ciega.csv','md4_plantilla_pps_todos_escenarios.csv',
        'md4_primera_ronda_segundo_evaluador.csv',
        'md4_segunda_ronda_si_corresponde.csv',
        'md4_recalibracion_10_pares_ajuste.csv',
        'md4_mapa_articulos_textos.csv',
        'md2_popularidad_articulos_desarrollo.csv',
        'md4_comparacion_topk_modelos.csv','md4_posiciones_pps.csv'}
    tamano_maximo = 20 * 1024 * 1024  # por fichero, nunca intenta incluir bases enormes
    incluidos, excluidos = [], []

    def sha256_archivo(p):
        h = hashlib.sha256()
        with p.open('rb') as fd:
            for bloque in iter(lambda: fd.read(4 * 1024 * 1024), b''):
                h.update(bloque)
        return h.hexdigest()

    candidatos = sorted(p for p in carpeta.rglob('*') if p.is_file())
    with zipfile.ZipFile(zip_destino, 'w', compression=zipfile.ZIP_DEFLATED,
                         compresslevel=5, allowZip64=True) as zf:
        for f in candidatos:
            rel = f.relative_to(carpeta)
            motivo = None
            if f.is_symlink(): motivo = 'ENLACE_SIMBOLICO'
            elif any(x in directorios_no_publicables for x in rel.parts[:-1]): motivo = 'CACHE_INTERNO'
            elif f.name in archivos_excluidos: motivo = 'IDENTIFICADORES_O_FINGERPRINTS_NO_PUBLICABLES'
            elif f.suffix.lower() not in extensiones: motivo = 'FORMATO_NO_EVIDENCIA'
            elif f.stat().st_size > tamano_maximo: motivo = 'ARCHIVO_DEMASIADO_GRANDE'
            if motivo:
                excluidos.append({'archivo':str(rel),'bytes':f.stat().st_size,'motivo':motivo})
                continue
            incluido = {'archivo':str(rel),'bytes':f.stat().st_size,
                        'sha256':sha256_archivo(f)}
            zf.write(f, arcname=f'{objetivo}/{rel.as_posix()}')
            incluidos.append(incluido)
        manifiesto = {
            'objetivo':objetivo, 'creado_utc':datetime.now(timezone.utc).isoformat(),
            'resultado_no_equivale_a_experimento_completo':True,
            'archivos_incluidos':incluidos,'archivos_excluidos':excluidos,
            'nota':'Estado de pruebas y resultados deben auditarse; el ZIP no acredita por si solo ejecucion completa.'
        }
        zf.writestr(f'{objetivo}/MANIFIESTO_ZIP.json', json.dumps(manifiesto,indent=2,ensure_ascii=False))
    with zipfile.ZipFile(zip_destino) as control:
        assert control.testzip() is None, 'ZIP corrupto'
        assert len(control.namelist()) == len(incluidos)+1
    return zip_destino,manifiesto

# Si OUT es la carpeta de este objetivo, se reutiliza; de otro modo usa la ruta canónica.
carpeta_actual = globals().get('OUT')
if carpeta_actual is not None and Path(carpeta_actual).name != OBJETIVO:
    carpeta_actual = None
zip_creado, resumen_zip = exportar_evidencias(OBJETIVO, carpeta_actual)
print('ZIP:', zip_creado)
print('Publicables:',len(resumen_zip['archivos_incluidos']),
      '| Excluidos:',len(resumen_zip['archivos_excluidos']))
print('Tamaño:',round(zip_creado.stat().st_size/1024**2,2),'MiB')
print('No se incluyen Parquet, caches, modelos binarios ni identificadores de clientes.')
display(FileLink(str(zip_creado)))


ZIP: /kaggle/working/HM_MD4_Evidencias.zip
Publicables: 23 | Excluidos: 13
Tamaño: 0.1 MiB
No se incluyen Parquet, caches, modelos binarios ni identificadores de clientes.


/kaggle/working/HM_MD4_Evidencias.zip